# Export PhysiCell Uptake CSVs

This notebook extracts the PhysiCell uptake-study outputs using the same microenvironment processing approach as the uptake comparison notebook and writes one CSV into each uptake experiment folder.

Each CSV contains the fields needed later by the plotting workflow: `tool`, `run_id`, `dx_um`, `dt_min`, `average_source`, `center_source`, `resolution_label`, `time_min`, `average_uM`, and `center_uM`.

The `resolution_label` column is kept because the downstream comparison notebooks still use it directly in grouping and plot labels.

In [73]:
from itertools import product
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
from pctk import multicellds

In [74]:
ROOT = Path('/home/tntiniak/Work/observatory_benchmark')
PHYSICELL_DIR = ROOT / 'PhysiCell' / 'results' / 'decay_study'
OUTPUT_FILENAME = 'physicell_uptake_plot_data.csv'
EXPERIMENT_FOLDERS = [
    PHYSICELL_DIR / 'size_5_uptake',
    PHYSICELL_DIR / 'size_10_uptake',
    PHYSICELL_DIR / 'size_20_uptake',
    PHYSICELL_DIR / 'size_40_uptake',
]

EXPERIMENT_FOLDERS

[PosixPath('/home/tntiniak/Work/observatory_benchmark/PhysiCell/results/decay_study/size_5_uptake'),
 PosixPath('/home/tntiniak/Work/observatory_benchmark/PhysiCell/results/decay_study/size_10_uptake'),
 PosixPath('/home/tntiniak/Work/observatory_benchmark/PhysiCell/results/decay_study/size_20_uptake'),
 PosixPath('/home/tntiniak/Work/observatory_benchmark/PhysiCell/results/decay_study/size_40_uptake')]

In [75]:
def make_physicell_resolution_label(dx_um: float) -> str:
    return f'voxel size={dx_um:.0f} um'


def load_physicell_run(folder: Path) -> pd.DataFrame:
    size_token = folder.name.split('_')[1]
    settings_path = folder / f'PhysiCell_settings_{size_token}_uptake.xml'
    dx_um = float(ET.parse(settings_path).getroot().findtext('.//domain/dx'))

    reader = multicellds.MultiCellDS(output_folder=str(folder))

    average_uM = []
    center_uM = []
    center_indices = None

    for _, microenvironment in reader.microenvironment_as_matrix_iterator():
        concentration_field = microenvironment[4]
        # print(microenvironment.shape)

        if center_indices is None:
            n_voxels = concentration_field.size
            # print(f'Number of voxels: {n_voxels}')
            n_per_dim = int(round(n_voxels ** (1 / 3)))
            # print(f'Number of voxels per dimension: {n_per_dim}')
            dims = (n_per_dim, n_per_dim, n_per_dim)

            midpoints = []
            for dim in dims:
                if dim % 2 == 1:
                    midpoints.append([dim // 2])
                else:
                    midpoints.append([dim // 2 - 1, dim // 2])

            center_coords = list(product(*midpoints))
            # print(f'Center coordinates: {center_coords}')
            center_indices = np.ravel_multi_index(np.array(center_coords).T, dims)
            # print(f'Center indices: {center_indices}')

        average_uM.append(concentration_field.mean() / 602.2)
        center_uM.append(concentration_field[center_indices].mean() / 602.2)

    time_min = np.round(np.arange(len(average_uM), dtype=float) * 0.1, 2)
    run_id = f'PhysiCell_dx_{int(dx_um)}um_uptake'

    return pd.DataFrame({
        'tool': 'PhysiCell',
        'run_id': run_id,
        'dx_um': dx_um,
        'dt_min': np.nan,
        'average_source': 'mean over all voxels',
        'center_source': 'mean over center voxels',
        'resolution_label': make_physicell_resolution_label(dx_um),
        'time_min': time_min,
        'average_uM': np.array(average_uM, dtype=float),
        'center_uM': np.array(center_uM, dtype=float),
    })

In [76]:
exported_files = []
preview_tables = []
row_counts = []

for folder in EXPERIMENT_FOLDERS:
    frame = load_physicell_run(folder)
    output_path = folder / OUTPUT_FILENAME
    frame.to_csv(output_path, index=False)
    exported_files.append(output_path)
    preview_tables.append(frame.head(3))
    row_counts.append(len(frame))

pd.DataFrame({
    'experiment_folder': [path.parent.name for path in exported_files],
    'csv_path': [str(path) for path in exported_files],
    'rows_written': row_counts,
})

,experiment_folder,csv_path,rows_written
0,size_5_uptake,/home/tntiniak/Work/observatory_benchmark/Phys...,101
1,size_10_uptake,/home/tntiniak/Work/observatory_benchmark/Phys...,101
2,size_20_uptake,/home/tntiniak/Work/observatory_benchmark/Phys...,101
3,size_40_uptake,/home/tntiniak/Work/observatory_benchmark/Phys...,101


In [77]:
pd.concat(preview_tables, ignore_index=True)

,tool,run_id,dx_um,dt_min,average_source,center_source,resolution_label,time_min,average_uM,center_uM
0,PhysiCell,PhysiCell_dx_5um_uptake,5.0,NaN,mean over all voxels,mean over center voxels,voxel size=5 um,0.0,10.000000,10.000000
1,PhysiCell,PhysiCell_dx_5um_uptake,5.0,NaN,mean over all voxels,mean over center voxels,voxel size=5 um,0.1,9.990663,9.350776
2,PhysiCell,PhysiCell_dx_5um_uptake,5.0,NaN,mean over all voxels,mean over center voxels,voxel size=5 um,0.2,9.981835,9.044263
3,PhysiCell,PhysiCell_dx_10um_uptake,10.0,NaN,mean over all voxels,mean over center voxels,voxel size=10 um,0.0,10.000000,10.000000
4,PhysiCell,PhysiCell_dx_10um_uptake,10.0,NaN,mean over all voxels,mean over center voxels,voxel size=10 um,0.1,9.957327,6.649302
5,PhysiCell,PhysiCell_dx_10um_uptake,10.0,NaN,mean over all voxels,mean over center voxels,voxel size=10 um,0.2,9.923727,5.875120
6,PhysiCell,PhysiCell_dx_20um_uptake,20.0,NaN,mean over all voxels,mean over center voxels,voxel size=20 um,0.0,10.000000,10.000000
7,PhysiCell,PhysiCell_dx_20um_uptake,20.0,NaN,mean over all voxels,mean over center voxels,voxel size=20 um,0.1,9.880855,4.455280
8,PhysiCell,PhysiCell_dx_20um_uptake,20.0,NaN,mean over all voxels,mean over center voxels,voxel size=20 um,0.2,9.817040,3.781712
9,PhysiCell,PhysiCell_dx_40um_uptake,40.0,NaN,mean over all voxels,mean over center voxels,voxel size=40 um,0.0,10.000000,10.000000


In [78]:
file = "/home/tntiniak/Work/observatory_benchmark/PhysiCell/results/decay_study/size_5_uptake/output00000000_microenvironment0.mat"
folder = Path(file).parent

In [79]:
from scipy.io import loadmat


s = loadmat(file)
s

settings_path = folder / f'PhysiCell_settings_5_uptake.xml'
dx_um = 5
reader = multicellds.MultiCellDS(output_folder=str(folder))

average_uM = []
center_uM = []
center_indices = None

for _, microenvironment in reader.microenvironment_as_matrix_iterator():
    concentration_field = microenvironment[4]
    # print(microenvironment[4].shape)
    
    # print(concentration_field)

    # if center_indices is None:
    #     n_voxels = concentration_field.size
    #     # print(f'Number of voxels: {n_voxels}')
    #     n_per_dim = int(round(n_voxels ** (1 / 3)))
    #     # print(f'Number of voxels per dimension: {n_per_dim}')
    #     dims = (n_per_dim, n_per_dim, n_per_dim)

    #     midpoints = []
    #     for dim in dims:
    #         if dim % 2 == 1:
    #             midpoints.append([dim // 2])
    #         else:
    #             midpoints.append([dim // 2 - 1, dim // 2])
    print(concentration_field.shape, concentration_field.min(), concentration_field.max())
    #     center_coords = list(product(*midpoints))
    #     # print(f'Center coordinates: {center_coords}')
    #     center_indices = np.ravel_multi_index(np.array(center_coords).T, dims)
    #     # print(f'Center indices: {center_indices}')

    average_uM.append(concentration_field.mean() / 602.2)
print(f'Average uM: {average_uM}')
    # center_uM.append(concentration_field[center_indices].mean() / 602.2)



(110592,) 6022.0 6022.0
(110592,) 0.05110111140426351 6022.000000000002
(110592,) 0.049542713058967354 6022.0
(110592,) 0.04870749122750391 6022.0
(110592,) 0.04817215016688374 6022.0
(110592,) 0.04779358857157771 6022.0
(110592,) 0.04750849657736998 6022.0
(110592,) 0.04728421069422636 6022.0
(110592,) 0.04710206019336879 6022.0
(110592,) 0.04695055946917489 6022.0
(110592,) 0.04682223968522012 6022.0
(110592,) 0.0467120247238662 6022.0
(110592,) 0.046616334770184197 6022.0
(110592,) 0.046532562361948526 6022.0
(110592,) 0.046458752208805715 6022.0
(110592,) 0.04639339858014611 6022.0
(110592,) 0.04633531345333262 6022.0
(110592,) 0.0462835386968371 6022.0
(110592,) 0.046237286392948274 6022.0
(110592,) 0.04619589752990373 6022.0
(110592,) 0.04615881289809238 6022.0
(110592,) 0.046125552217092226 6022.0
(110592,) 0.0460956988893524 6022.0
(110592,) 0.046068888649908095 6022.0
(110592,) 0.04604480094864782 6022.0
(110592,) 0.046023152274976026 6022.0
(110592,) 0.04600369088323925 6022.

In [80]:
average_uM

[np.float64(10.0),
 np.float64(9.990662756524383),
 np.float64(9.981835487422867),
 np.float64(9.973213082824689),
 np.float64(9.964827024113728),
 np.float64(9.956760767429948),
 np.float64(9.949098851176755),
 np.float64(9.941904226826674),
 np.float64(9.935213060727687),
 np.float64(9.92903778156367),
 np.float64(9.923372729332756),
 np.float64(9.918199762578793),
 np.float64(9.9134928789767),
 np.float64(9.909221687998306),
 np.float64(9.90535387361525),
 np.float64(9.901856863187799),
 np.float64(9.898698908914552),
 np.float64(9.895849750795445),
 np.float64(9.893280989696343),
 np.float64(9.890966264416788),
 np.float64(9.88888129954221),
 np.float64(9.887003870683024),
 np.float64(9.885313719126119),
 np.float64(9.883792437593376),
 np.float64(9.88242334157957),
 np.float64(9.881191335745525),
 np.float64(9.88008278141775),
 np.float64(9.879085368919263),
 np.float64(9.878187996891374),
 np.float64(9.877380659726366),
 np.float64(9.876654343550427),
 np.float64(9.87600093076054